# EDA — Sander's Panorama 2024 Shapefile

Exploratory notebook for the two files saved in `data/raw/Panorama_2024_RD/`:
- `Panorama_2024_RD/Panorama_2024_RD.shp` — shapefile with point geometry per photo
- `import_locations.txt` — tab-separated file with coordinates and camera metadata

Goal: understand the contents, quirks, and spatial coverage so we know **how to use these files
in notebook `04b_leg_photo_selection_shp.ipynb` instead of the old `selected_photos_near_intersections.csv`**.

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"

SHP_PATH = os.path.join(PROJECT_DIR, "data", "raw", "Panorama_2024_RD",
                        "Panorama_2024_RD", "Panorama_2024_RD.shp")

TXT_PATH = os.path.join(PROJECT_DIR, "data", "raw", "Panorama_2024_RD",
                        "import_locations.txt")

# Reference: intersections used in notebook 04b
INTERSECTIONS_PATH = os.path.join(PROJECT_DIR, "data", "processed",
                                   "intersections_stratified.gpkg")

# Old photo input that 04b currently uses (for comparison)
OLD_CSV_PATH = os.path.join(PROJECT_DIR, "data", "processed",
                             "selected_photos_near_intersections.csv")

CRS_RD = "EPSG:28992"  # RD New — Dutch national coordinate system

## 1. Shapefile

In [ ]:
# Load the shapefile.
# It is large (618k rows) but contains only point geometry plus a few attributes.
shp = gpd.read_file(SHP_PATH)

print(f"Rows    : {len(shp):,}")
print(f"Columns : {list(shp.columns)}")
print(f"CRS     : {shp.crs}")
print(f"Bounds  : {shp.total_bounds}")
print()
shp.head()

In [ ]:
# Check dtypes and whether X/Y columns match the geometry coordinates.
# The shapefile stores X and Y as separate float columns AND as geometry.
# We verify they agree so we know which one to trust later.
print("Dtypes:")
print(shp.dtypes)
print()

# Compute difference between the explicit X/Y columns and the geometry coordinates.
# Should be ~0 if they were exported from the same source.
diff_x = (shp.geometry.x - shp["X"]).abs().max()
diff_y = (shp.geometry.y - shp["Y"]).abs().max()
print(f"Max |geometry.x - X column| : {diff_x:.6f}  (should be ~0)")
print(f"Max |geometry.y - Y column| : {diff_y:.6f}  (should be ~0)")

In [ ]:
# Inspect the Filename format.
# Expected: 'recording_YYYY-MM-DD_HH-MM-SS_NNNNN'
# Extract the session ID (everything except the trailing frame number).
shp["session"] = shp["Filename"].str.extract(r"(recording_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")

print(f"Unique sessions (recording runs): {shp['session'].nunique()}")
print("\nTop 10 sessions by frame count:")
print(shp["session"].value_counts().head(10))

## 2. import_locations.txt

In [ ]:
# Load the tab-separated metadata file.
# Important: X, Y, Z, Pan, Tilt, Roll use a COMMA as decimal separator (European locale).
# We read them as strings first, then convert.
txt = pd.read_csv(TXT_PATH, sep="\t")

print(f"Rows    : {len(txt):,}")
print(f"Columns : {list(txt.columns)}")
print()
print("Dtypes (raw, before decimal-comma fix):")
print(txt.dtypes)
print()
txt.head()

In [ ]:
# Fix decimal comma → decimal point for all float columns.
# Pandas reads these as 'object' (string) because '98448,887' is not a valid float.
float_cols = ["X", "Y", "Z", "Pan", "Tilt", "Roll"]
for col in float_cols:
    txt[col] = txt[col].astype(str).str.replace(",", ".").astype(float)

print("Dtypes after fix:")
print(txt.dtypes)
print()
txt.head()

In [ ]:
# Descriptive statistics for coordinate and camera columns.
txt[["X", "Y", "Z", "Pan", "Tilt", "Roll"]].describe()

In [ ]:
# Key question: does Pan contain actual camera heading, or is it always 0?
# In the old 04b, Pan = 0 for all photos (north-facing convention).
# If that holds here too, we cannot use Pan as a bearing filter.
print("Pan value counts:")
print(txt["Pan"].value_counts())
print()
print(f"Pan is always 0: {(txt['Pan'] == 0).all()}")

In [ ]:
# Timestamp: Unix epoch seconds.
# Convert to human-readable datetime so we know when photos were taken.
txt["datetime"] = pd.to_datetime(txt["Timestamp"], unit="s", utc=True)
print(f"Date range: {txt['datetime'].min()}  →  {txt['datetime'].max()}")

## 3. Join shapefile and txt on Filename

In [ ]:
# The shapefile has geometry; the txt has Z, Pan, Tilt, Roll, Timestamp.
# Both have the same 618,600 rows and the same Filename column as join key.
# We verify the join is exact (no orphan rows) before recommending it for 04b.

shp_names = set(shp["Filename"])
txt_names = set(txt["Filename"])

print(f"Filenames only in shapefile : {len(shp_names - txt_names)}")
print(f"Filenames only in txt       : {len(txt_names - shp_names)}")
print(f"Filenames in both           : {len(shp_names & txt_names):,}")
print()

# Also confirm X/Y values match after parsing (different decimal separator encoding).
merged_check = shp[["Filename", "X", "Y"]].merge(
    txt[["Filename", "X", "Y"]].rename(columns={"X": "X_txt", "Y": "Y_txt"}),
    on="Filename", how="inner"
)
diff_x2 = (merged_check["X"] - merged_check["X_txt"]).abs().max()
diff_y2 = (merged_check["Y"] - merged_check["Y_txt"]).abs().max()
print(f"Max |X_shp - X_txt| after join : {diff_x2:.6f}")
print(f"Max |Y_shp - Y_txt| after join : {diff_y2:.6f}")

In [ ]:
# Build the merged GeoDataFrame — the full dataset as it would be used in 04b.
# We take the geometry from the shapefile and add Z/Pan/Tilt/Roll/Timestamp from the txt.
# Drop the redundant X_txt/Y_txt since geometry already carries coordinates.

photos = shp.merge(
    txt[["Filename", "Z", "Pan", "Tilt", "Roll", "Timestamp"]],
    on="Filename",
    how="inner"
)

print(f"Merged GeoDataFrame rows : {len(photos):,}")
print(f"Columns                  : {list(photos.columns)}")
photos.head()

## 4. Spatial coverage

In [ ]:
# Load the intersections used in 04b so we can compare coverage.
intersections = gpd.read_file(INTERSECTIONS_PATH)

print(f"Intersections count : {len(intersections):,}")
print(f"Intersections CRS   : {intersections.crs}")
print(f"Intersections bounds: {intersections.total_bounds}")
print()
print(f"Shapefile bounds    : {photos.total_bounds}")

In [ ]:
# How many shapefile photos fall within the intersection bounding box?
# This gives a rough sense of how much of the 618k dataset is useful for 04b.
xmin, ymin, xmax, ymax = intersections.total_bounds

in_bbox = photos[
    (photos["X"] >= xmin) & (photos["X"] <= xmax) &
    (photos["Y"] >= ymin) & (photos["Y"] <= ymax)
]

print(f"Photos inside intersection bounding box : {len(in_bbox):,} / {len(photos):,}")
print(f"That is {len(in_bbox)/len(photos)*100:.1f}% of the full dataset")

In [ ]:
# Visual comparison: plot photo points and intersection locations together.
fig, ax = plt.subplots(figsize=(10, 10))

# Sample photos to keep the plot fast
sample = photos.sample(min(5000, len(photos)), random_state=42)

sample.plot(ax=ax, markersize=0.5, color="steelblue", alpha=0.3, label="Photos (sample)")
intersections.plot(ax=ax, markersize=8, color="red", alpha=0.7, label="Intersections")

ax.set_title("Spatial coverage: shapefile photos vs. intersections (RD New)")
ax.set_xlabel("X (RD New)")
ax.set_ylabel("Y (RD New)")

blue_patch = mpatches.Patch(color="steelblue", label="Photos (5k sample)")
red_patch  = mpatches.Patch(color="red",       label="Intersections")
ax.legend(handles=[blue_patch, red_patch])

plt.tight_layout()
plt.show()

## 5. Comparison with the old dataset currently used in 04b

In [ ]:
# The current 04b input is selected_photos_near_intersections.csv produced by notebook 02b.
# That file comes from a completely different source panorama dataset (old Recording.csv).
# Let's compare the two side by side.
old = pd.read_csv(OLD_CSV_PATH)

print("=== Old dataset (selected_photos_near_intersections.csv) ===")
print(f"Rows    : {len(old):,}")
print(f"Columns : {list(old.columns)}")
print(f"X range : {old['X/Long'].min():.1f} – {old['X/Long'].max():.1f}")
print(f"Y range : {old['Y/Lat'].min():.1f} – {old['Y/Lat'].max():.1f}")
print(f"Filename sample: {old['Filename'].iloc[0]}")
print()
print("=== New dataset (Panorama_2024_RD shapefile + txt) ===")
print(f"Rows    : {len(photos):,}")
print(f"Columns : {list(photos.columns)}")
print(f"X range : {photos['X'].min():.1f} – {photos['X'].max():.1f}")
print(f"Y range : {photos['Y'].min():.1f} – {photos['Y'].max():.1f}")
print(f"Filename sample: {photos['Filename'].iloc[0]}")

In [ ]:
# The old dataset already has a Pan column (from notebook 02b's merge with Recording.csv).
# Check whether it also uses Pan = 0 (north-facing convention).
if "Pan" in old.columns:
    print("Old dataset Pan stats:")
    print(old["Pan"].describe())
else:
    print("Old dataset has no Pan column — it is added later in 04b logic.")

## 6. Conclusions and recommendations for 04b

### What the new files contain

| File | Rows | Columns | Notes |
|---|---|---|---|
| `Panorama_2024_RD.shp` | 618,600 | `Filename`, `X`, `Y`, `geometry` | EPSG:28992. No camera orientation data. |
| `import_locations.txt` | 618,600 | `Filename`, `X`, `Y`, `Z`, `Pan`, `Tilt`, `Roll`, `Timestamp` | X/Y/Z use **comma as decimal separator**. Pan = 0 everywhere. |

### How to merge them
`Filename` is the join key. All filenames appear in both files with no orphan rows.
The preferred merge is: take **geometry from the shapefile** and **Z / Pan / Tilt / Roll / Timestamp from the txt**.

### Pan = 0 — same north-facing convention as before
`Pan` is 0.0 for every row. This is identical to the convention already assumed in `04b`
(where notebook `02b` sets `Pan = 0` as a constant). **No change to the bearing logic is needed.**

### Column name differences vs. the old dataset

| Old (`selected_photos_near_intersections.csv`) | New (shapefile + txt) |
|---|---|
| `X/Long` | `X` |
| `Y/Lat` | `Y` |
| `Filename` | `Filename` |
| `Timestamp` | `Timestamp` |
| *(no Z, Tilt, Roll)* | `Z`, `Tilt`, `Roll` (unused in 04b) |

The only change needed in `04b` is to:
1. Replace the `SELECTED_CSV` load with a load of the merged GeoDataFrame above.
2. Rename columns `X` → `X/Long` and `Y` → `Y/Lat` (or update references in the notebook).
3. Ensure `Pan` column is present — it is, as a constant 0.

### Spatial coverage
The new dataset covers a larger area than just Rotterdam. Filtering to the intersection bounding box
reduces the dataset to only what is needed — this can be done as a spatial join or a simple bbox clip.
Notebook 02b's pre-filtering step (selecting photos near intersections) should be replicated for the new data.